# 06 · Unrestricted Hartree-Fock (UHF) with DIIS

UHF 放松 RHF 的闭壳层约束，让 alpha 与 beta 电子使用不同的空间轨道。

本 notebook 沿用 `02-rhf/rhf-diis.ipynb` 的结构：先构造积分和密度矩阵，再实现 J/K、能量、DIIS，最后与 PySCF 对照。

---

## 1 · UHF 方程

UHF 有两套 Roothaan 方程：

$$
\mathbf{F}^{\alpha}\mathbf{C}^{\alpha}
=
\mathbf{S}\mathbf{C}^{\alpha}\boldsymbol{\varepsilon}^{\alpha},
\qquad
\mathbf{F}^{\beta}\mathbf{C}^{\beta}
=
\mathbf{S}\mathbf{C}^{\beta}\boldsymbol{\varepsilon}^{\beta}
$$

密度矩阵也分成两块：

$$
\mathbf{D}^{\alpha}_{\mu\nu}
=
\sum_i^{N_\alpha} C^\alpha_{\mu i} C^{\alpha *}_{\nu i},
\qquad
\mathbf{D}^{\beta}_{\mu\nu}
=
\sum_i^{N_\beta} C^\beta_{\mu i} C^{\beta *}_{\nu i}
$$

Coulomb 势来自总密度，exchange 只在同自旋电子之间出现：

$$
\mathbf{F}^{\alpha}
=
\mathbf{H}^{\mathrm{core}}
+
\mathbf{J}[\mathbf{D}^{\alpha}+\mathbf{D}^{\beta}]
-
\mathbf{K}[\mathbf{D}^{\alpha}]
$$

$$
\mathbf{F}^{\beta}
=
\mathbf{H}^{\mathrm{core}}
+
\mathbf{J}[\mathbf{D}^{\alpha}+\mathbf{D}^{\beta}]
-
\mathbf{K}[\mathbf{D}^{\beta}]
$$

---

## 2 · 分子设定与积分

这里用 OH radical doublet 测试 UHF。PySCF 中 `spin = N_alpha - N_beta = 1`。

In [ ]:
from pyscf import gto
import numpy as np
from scipy.linalg import eigh, fractional_matrix_power

# -------- 分子定义: OH radical / cc-pVDZ --------
mol = gto.M(
    atom="O 0 0 0; H 0 0 0.9697",
    basis="cc-pvdz",
    spin=1,
    verbose=0,
)
nao = mol.nao_nr()
nelec = mol.nelectron
nalpha, nbeta = mol.nelec

# -------- 积分 --------
S = mol.intor("int1e_ovlp_sph")
T = mol.intor("int1e_kin_sph")
V = mol.intor("int1e_nuc_sph")
H_core = T + V
A = fractional_matrix_power(S, -0.5)
eri = mol.intor("int2e")
E_nn = mol.energy_nuc()

print(f"nao = {nao}, nelec = {nelec}, nalpha = {nalpha}, nbeta = {nbeta}")
print(f"S shape = {S.shape}, H_core shape = {H_core.shape}")
print(f"eri shape = {eri.shape} ({eri.size:,} elements)")
print(f"E_nn = {E_nn:.10f} Hartree")

---

## 3 · SCF 核心函数

In [ ]:
# ============================================================
# 3.1 占据数: 每个 spin orbital 最多占 1 个电子
# ============================================================
def get_occ(mo_energy, nocc):
    e_idx = np.argsort(mo_energy)
    mo_occ = np.zeros_like(mo_energy)
    mo_occ[e_idx[:nocc]] = 1
    return mo_occ


# ============================================================
# 3.2 密度矩阵: D = C_occ C_occ^dagger
# ============================================================
def make_rdm1(mo_coeff, mo_occ):
    mocc = mo_coeff[:, mo_occ > 0]
    occ = mo_occ[mo_occ > 0]
    return (mocc * occ).dot(mocc.conj().T)


# ============================================================
# 3.3 J 和 K 矩阵
# ============================================================
def get_jk(eri, dm):
    vj = np.einsum("ijkl,kl->ij", eri, dm, optimize=True)
    vk = np.einsum("ikjl,kl->ij", eri, dm, optimize=True)
    return vj, vk


# ============================================================
# 3.4 UHF 有效势
# ============================================================
def get_veff(eri, dm_alpha, dm_beta):
    dm_total = dm_alpha + dm_beta
    vj, _ = get_jk(eri, dm_total)
    _, vk_alpha = get_jk(eri, dm_alpha)
    _, vk_beta = get_jk(eri, dm_beta)
    return vj - vk_alpha, vj - vk_beta


# ============================================================
# 3.5 Fock 矩阵
# ============================================================
def get_fock(H_core, V_eff):
    return H_core + V_eff


# ============================================================
# 3.6 电子能量
# E_elec = Tr[(Da + Db) H] + 1/2 Tr[Da Va + Db Vb]
# ============================================================
def energy_elec(dm_alpha, dm_beta, H_core, V_eff_alpha, V_eff_beta):
    dm_total = dm_alpha + dm_beta
    e1 = np.einsum("ij,ji->", H_core, dm_total, optimize=True)
    e2_alpha = np.einsum("ij,ji->", V_eff_alpha, dm_alpha, optimize=True)
    e2_beta = np.einsum("ij,ji->", V_eff_beta, dm_beta, optimize=True)
    e2 = 0.5 * (e2_alpha + e2_beta)
    return (e1 + e2).real, e2.real


def energy_tot(dm_alpha, dm_beta, H_core, V_eff_alpha, V_eff_beta):
    return E_nn + energy_elec(dm_alpha, dm_beta, H_core, V_eff_alpha, V_eff_beta)[0]


# ============================================================
# 3.7 初猜密度: 对角化 H_core，alpha/beta 分别填充
# ============================================================
def get_init_guess(H_core, S):
    mo_energy, mo_coeff = eigh(H_core, S)
    mo_occ_alpha = get_occ(mo_energy, nalpha)
    mo_occ_beta = get_occ(mo_energy, nbeta)
    dm_alpha = make_rdm1(mo_coeff, mo_occ_alpha)
    dm_beta = make_rdm1(mo_coeff, mo_occ_beta)
    return dm_alpha, dm_beta


print("UHF helper functions ready.")

---

## 4 · DIIS

UHF 的 DIIS 误差向量分别计算 alpha/beta 两块，再把它们一起放进同一个 Pulay 方程。

In [ ]:
def get_err_vec(fock_alpha, fock_beta, dm_alpha, dm_beta, S, A):
    err_alpha = A @ (fock_alpha @ dm_alpha @ S - S @ dm_alpha @ fock_alpha) @ A
    err_beta = A @ (fock_beta @ dm_beta @ S - S @ dm_beta @ fock_beta) @ A
    return np.array([err_alpha, err_beta])


def apply_diis(fock_list, err_vec_list):
    n = len(fock_list)
    dim = n + 1
    B = np.empty((dim, dim))
    B[-1, :] = -1
    B[:, -1] = -1
    B[-1, -1] = 0

    for i in range(n):
        for j in range(n):
            B[i, j] = np.vdot(err_vec_list[i], err_vec_list[j]).real

    rhs = np.zeros(dim)
    rhs[-1] = -1

    try:
        coeff = np.linalg.solve(B, rhs)[:-1]
    except np.linalg.LinAlgError:
        coeff = np.linalg.lstsq(B, rhs, rcond=None)[0][:-1]

    return np.einsum("i,ixyz->xyz", coeff, np.asarray(fock_list), optimize=True)


print("DIIS functions defined.")

---

## 5 · UHF + DIIS 完整运行

In [ ]:
max_cycle = 80
max_diis = 8
conv_tol = 1e-10
conv_tol_dm = 1e-8

e_old = 0.0
fock_list = []
err_vec_list = []

dm_alpha, dm_beta = get_init_guess(H_core, S)

print(f"{'Iter':>4s}  {'E_total':>16s}  {'dE':>10s}  {'dD':>10s}")
print("-" * 46)

converged = False
for cycle in range(max_cycle):
    V_eff_alpha, V_eff_beta = get_veff(eri, dm_alpha, dm_beta)
    fock_alpha = get_fock(H_core, V_eff_alpha)
    fock_beta = get_fock(H_core, V_eff_beta)
    e_tot = energy_tot(dm_alpha, dm_beta, H_core, V_eff_alpha, V_eff_beta)

    err = get_err_vec(fock_alpha, fock_beta, dm_alpha, dm_beta, S, A)
    fock_list.append(np.array([fock_alpha, fock_beta]))
    err_vec_list.append(err)
    if len(fock_list) > max_diis:
        fock_list.pop(0)
        err_vec_list.pop(0)

    if cycle >= 2:
        fock_alpha, fock_beta = apply_diis(fock_list, err_vec_list)

    mo_energy_alpha, mo_coeff_alpha = eigh(fock_alpha, S)
    mo_energy_beta, mo_coeff_beta = eigh(fock_beta, S)

    mo_occ_alpha = get_occ(mo_energy_alpha, nalpha)
    mo_occ_beta = get_occ(mo_energy_beta, nbeta)
    dm_alpha_new = make_rdm1(mo_coeff_alpha, mo_occ_alpha)
    dm_beta_new = make_rdm1(mo_coeff_beta, mo_occ_beta)

    e_diff = abs(e_tot - e_old)
    d_norm = np.linalg.norm(dm_alpha_new - dm_alpha) + np.linalg.norm(dm_beta_new - dm_beta)

    print(f"{cycle+1:4d}  {e_tot:16.10f}  {e_diff:10.3e}  {d_norm:10.3e}")

    if e_diff < conv_tol and d_norm < conv_tol_dm:
        converged = True
        print(f"\nConverged in {cycle+1} iterations!")
        print(f"E_total (UHF+DIIS/cc-pVDZ) = {e_tot:.10f} Hartree")
        print(f"E_elec = {e_tot - E_nn:.10f}")
        print(f"E_nn   = {E_nn:.10f}")
        break

    dm_alpha, dm_beta = dm_alpha_new, dm_beta_new
    e_old = e_tot

if not converged:
    print("WARNING: UHF SCF did not converge!")

---

## 6 · 与 PySCF 对照

In [ ]:
from pyscf import scf

mf_ref = scf.UHF(mol)
mf_ref.conv_tol = conv_tol
mf_ref.kernel(dm0=np.array([dm_alpha, dm_beta]))

print(f"\nPySCF UHF:     {mf_ref.e_tot:.10f} Hartree")
print(f"Our UHF+DIIS:   {e_tot:.10f} Hartree")
print(f"Difference:     {abs(mf_ref.e_tot - e_tot):.2e} Hartree")

ss, multiplicity = mf_ref.spin_square()
print(f"<S^2> = {ss:.8f}, 2S+1 = {multiplicity:.8f}")

---

## 7 · 总结

| 模块 | UHF 形式 |
|------|----------|
| 密度 | $\mathbf{D}^{\alpha}$ 与 $\mathbf{D}^{\beta}$ 分开构造 |
| Coulomb | $\mathbf{J}$ 使用总密度 $\mathbf{D}^{\alpha}+\mathbf{D}^{\beta}$ |
| Exchange | $\mathbf{K}^{\alpha}$ / $\mathbf{K}^{\beta}$ 只使用同自旋密度 |
| 能量 | $E = \mathrm{Tr}[(D^\alpha+D^\beta)H] + \frac12\mathrm{Tr}[D^\alpha V^\alpha + D^\beta V^\beta] + E_{nn}$ |
| DIIS | alpha/beta 两个误差块共同进入同一个 DIIS 外推 |